# settlement_loader — exploratory tests

Run each cell in order. The loader writes to `data/warehouse.duckdb` (created automatically).

In [ ]:
import sys
sys.path.insert(0, '..')  # make src/ importable from tests/

import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s  %(message)s')

## 1 — First load

In [ ]:
from src.ingest.settlement_loader import load

CSV = '../docs/sample-data/settlement_paysettler.csv'

n = load(CSV)
print(f'Rows processed: {n}')

## 2 — Idempotency check

Running twice must not duplicate rows in the table.

In [ ]:
n2 = load(CSV)
print(f'Second load: {n2} rows processed')

from src.db import get_connection
conn = get_connection()
total = conn.execute('SELECT COUNT(*) FROM raw_paysettler_settlements').fetchone()[0]
print(f'Total rows in table: {total}  (expected ≤ {n} due to deduplication by PK)')
conn.close()

## 3 — Row counts and status breakdown

In [ ]:
conn = get_connection()
conn.execute("""
    SELECT
        status,
        COUNT(*)                       AS total_rows,
        SUM(amount)                    AS total_amount,
        MIN(settled_at::DATE)          AS earliest_date,
        MAX(settled_at::DATE)          AS latest_date
    FROM raw_paysettler_settlements
    GROUP BY status
    ORDER BY status
""").df()

## 4 — Amount normalization

The CSV ships dirty values like `R$ 32.245,91`. Confirm they were parsed correctly.

In [ ]:
# All amounts must be positive
bad = conn.execute("""
    SELECT COUNT(*) FROM raw_paysettler_settlements WHERE amount <= 0
""").fetchone()[0]
print(f'Rows with amount <= 0: {bad}  (expected 0)')

# Amount distribution
conn.execute("""
    SELECT
        MIN(amount)  AS min_amount,
        MAX(amount)  AS max_amount,
        AVG(amount)  AS avg_amount
    FROM raw_paysettler_settlements
""").df()

## 5 — Timestamp normalization

The CSV has two date formats (`+00:00` and `Z`). Both must land as valid TIMESTAMPTZ.

In [ ]:
null_dates = conn.execute("""
    SELECT COUNT(*) FROM raw_paysettler_settlements WHERE settled_at IS NULL
""").fetchone()[0]
print(f'NULL settled_at: {null_dates}  (expected 0)')

conn.execute("""
    SELECT
        settled_at::DATE AS date,
        COUNT(*)         AS transactions
    FROM raw_paysettler_settlements
    GROUP BY date
    ORDER BY date
""").df()

## 6 — Duplicate transaction_ids in source CSV

The CSV itself has 510 rows but 10 duplicate `transaction_id`s — the PK resolves them to 500 unique rows.

In [ ]:
import duckdb
raw = duckdb.connect()
raw.execute("""
    SELECT transaction_id, COUNT(*) AS occurrences
    FROM read_csv_auto('../docs/sample-data/settlement_paysettler.csv', header=true)
    GROUP BY transaction_id
    HAVING occurrences > 1
    ORDER BY occurrences DESC
""").df()

## 7 — Sample rows

In [ ]:
conn.execute("""
    SELECT * FROM raw_paysettler_settlements LIMIT 10
""").df()

In [ ]:
conn.close()
raw.close()